# SuperStore

In [ ]:
import pandas as pd
import duckdb

from modules.const import DUCKDB_FILE_PATH

with duckdb.connect(DUCKDB_FILE_PATH) as conn:
    df = conn.sql("SELECT * FROM FactSales").df()
    conn.sql("DESCRIBE FactSales").show()

df.dtypes

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Row_ID      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ OrderID     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ OrderDate   │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ShipDate    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ShipMode    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ProductKey  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ CustomerKey │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ GeoKey      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Sales       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



Row_ID                  int64
OrderID                   str
OrderDate      datetime64[us]
ShipDate       datetime64[us]
ShipMode                  str
ProductKey            float64
CustomerKey             int64
GeoKey                  int64
Sales                 float64
dtype: object

## 1. YoY Revenue growth per Category  

Revenue per anno e categoria,  
variazione percentuale anno su anno.

| Colonna        | Descrizione |
|---             | ---|
| `year`         | Anno dell'ordine |
| `category`     | Categoria prodotto |
| `revenue`      | Revenue totale dell'anno |
| `prev_revenue` | Revenue anno precedente |
| `yoy_pct`      | Variazione % su anno precedente |

## 2. Customer cohort retention  

Raggruppamento di clienti per anno del primo acquisto (cohort),  
e misura del revenue generato da quella cohort negli anni successivi. 

| Colonna                    | Descrizione |
|---                         | ---|
| `cohort_year`              | Anno del primo acquisto del cliente |
| `order_year`               | Anno in cui la cohort ha generato revenue |
| `periods_since_first`      | Distanza in anni tra `order_year` e `cohort_year` |
| `customers`                | Clienti distinti attivi in quell'anno |
| `revenue`                  | Revenue totale della cohort in quell'anno |
| `avg_revenue_per_customer` | Revenue medio per cliente attivo |

## 3. ABC analysis sui prodotti  

Ordinamento dei prodotti per revenue decrescente,  
calcolo del revenue cumulato,  
classifica in A (primo 80%), B (80–95%), C (coda).

| Colonna          | Descrizione |
|---               | ---|
| `product_id`     | Identificativo prodotto |
| `product_name`   | Nome prodotto |
| `category`       | Categoria |
| `subcategory`    | Sottocategoria |
| `revenue`        | Revenue totale del prodotto |
| `revenue_pct`    | Percentuale sul revenue totale |
| `cumulative_pct` | Percentuale cumulata (ordinata per revenue desc) |
| `class`          | Classificazione: `A` (0–80%), `B` (80–95%), `C` (95–100%) |

## 4. Shipping performance  

Calcolo delay effettivo (`ShipDate` − `OrderDate`) per ogni ShipMode,  
confronto con il delay atteso (codificato in `modules.randomizer.Randomizer`, vd sotto),  
e isolamento ordini anomali oltre la soglia.

`Same Day`:       range(0, 2)  
`Standard Class`: range(4, 8)  
`First Class`:    range(1, 4)  
`Second Class`:   range(2, 6)  

| Colonna          | Descrizione |
|---               | ---|
| `ship_mode`      | Modalità di spedizione |
| `orders`         | Numero ordini |
| `avg_delay_days` | Delay medio effettivo in giorni (ShipDate − OrderDate) |
| `min_delay_days` | Delay minimo |
| `max_delay_days` | Delay massimo |
| `expected_min`   | Delay minimo atteso per quel ShipMode |
| `expected_max`   | Delay massimo atteso per quel ShipMode |
| `anomalies`      | Ordini fuori dalla finestra attesa |
| `anomaly_pct`    | Percentuale di ordini anomali |

## 5. Geographic market penetration  

Revenue e numero ordini per Stato,  
con in aggiunta il revenue medio per ordine  
e il rank nazionale.  
Identificare Stati ad alto volume ma basso ticket medio (potenziale upselling) e viceversa.  

| Colonna                | Descrizione |
|---                     | ---|
| `region`               | Regione |
| `state`                | Stato |
| `orders`               | Numero ordini distinti |
| `revenue`              | Revenue totale |
| `avg_order_value`      | Revenue medio per ordine |
| `national_revenue_pct` | Percentuale sul revenue nazionale |
| `revenue_rank`         | Rank per revenue all'interno della regione |
| `segment`              | `high_volume_low_ticket` / `low_volume_high_ticket` / `other` |